In [15]:
import os, shutil, re
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

In [16]:
def get_yaml_params(yaml_file):
    params = {}
    with open(yaml_file) as file:
        lines = file.readlines()
        values = [line.split(':')[1].strip() for line in lines if line.find(':')!=-1]
        keys = [line.split(':')[0] for line in lines if line.find(':')!=-1]
    return {keys[i]:values[i] for i in range(len(keys))}

def get_png_area(png_file, resolution):
    img = Image.open(png_file).convert('L')
    matrice = np.array(img)
    return len(matrice[matrice==254])*(resolution**2)

get_yaml_params('ros-exploration-config/output/E34-2/run1/Maps/2.0Map.yaml')

{'image': '/root/catkin_ws/src/my_navigation_configs/runs/outputs/E34-2/run1/Maps/2.0Map.pgm',
 'resolution': '0.030000',
 'origin': '[-50.010000, -50.010000, 0.000000]',
 'negate': '0',
 'occupied_thresh': '0.65',
 'free_thresh': '0.196'}

In [22]:
path = 'ros-exploration-config/output'
names = os.listdir(path)
already_computed = os.listdir('dataset')
count = 0
for name in names:
    maps_names = os.listdir(f'{path}/{name}/run1/Maps')
    if name in already_computed:
        continue
    if any((n.find('APE.yaml')!=-1 for n in maps_names)):
        count += 1
        os.makedirs(f'dataset/{name}',exist_ok=True)
        for n in maps_names:
            if n.endswith('APE.yaml'):
                nr = re.findall('[0-9]+',n)[0]
                shutil.copy(
                    f'{path}/{name}/run1/Maps/{n}',
                    f'dataset/{name}'
                )
                shutil.copy(
                    f'{path}/{name}/run1/Maps/{nr}{".0" if int(nr) else ""}Map.png',
                    f'dataset/{name}'
                )
print(count)

309


In [23]:
maps_names = os.listdir(f'dataset')
count = 0
for c in maps_names:
    count += len(os.listdir(f'dataset/{c}'))
count//2

4732

In [ ]:
file_path = 'dataset/W92-2/9APE.yaml'
if os.path.exists(file_path):
    print(f"Il file {file_path} esiste.")
else:
    print(f"Il file {file_path} non esiste.")

In [25]:
path = 'ros-exploration-config/output'
names = os.listdir(path)
for name in names:
    maps_names = os.listdir(f'{path}/{name}/run1/Maps')
    if all((n.find('APE.yaml')==-1 for n in maps_names)): continue
    for n in maps_names:
        full_path = f'{path}/{name}/run1/Maps'
        if n.endswith('Map.yaml') and n!='Map.yaml':
            nr = re.findall('[0-9]+',n)[0]
            if not os.path.exists(f'{full_path}/{nr}APE.yaml'): continue
            pre = n[:-5]
            area = get_png_area(
                f'{full_path}/{pre}.png',
                float(get_yaml_params(f'{full_path}/{n}')['resolution'])
            )
            with open(f'dataset/{name}/{re.findall(r'[0-9]+',pre)[0]}APE.yaml','a') as APE:
                APE.write(f'\narea: {area}')

## Bounding square

In [26]:
def bounding_square(matrix_img):
    matrix = np.array(matrix_img)
    h,w = len(matrix),len(matrix[0])
    # 0 walls, 205 background, 254 free area
    min_x = 0
    finded = False
    while not finded:
        if min_x==w-1: break
        finded = np.any(matrix[:,min_x:min_x+1]!=205)
        min_x += 1
    min_y = 0
    finded = False
    while not finded:
        if min_y==h-1: break
        finded = np.any(matrix[min_y:min_y+1,:]!=205)
        min_y += 1
    max_x = len(matrix[0])-1
    finded = False
    while not finded:
        if max_x==0: break
        finded = np.any(matrix[:,max_x-1:max_x]!=205)
        max_x -= 1
    max_y = len(matrix)-1
    finded = False
    while not finded:
        if max_y==0: break
        finded = np.any(matrix[max_y-1:max_y,:]!=205)
        max_y -= 1
    center_x, center_y = (max_x+min_x)//2,(max_y+min_y)//2
    side1, side2 = np.abs(max_x-min_x), np.abs(max_y-min_y)
    square_side = max((side1,side2))
    half = square_side//2+2
    move_x, move_y = 0,0
    if center_y-half<0: move_y = -(center_y-half)
    elif center_y+half>=h: move_y = h-(center_y+half)
    if center_x-half<0: move_x = -(center_x-half)
    elif center_x+half>=w: move_x = w-(center_x+half)
    return matrix[center_y-half+move_y:center_y+half+move_y,center_x-half+move_x:center_x+half+move_x]

In [27]:
Image.fromarray(bounding_square(np.array(Image.open('dataset/1-0/0Map.png')))).save('bo.png')

In [28]:
run_names = os.listdir('dataset')

for run in run_names:
    path = f'dataset/{run}'
    n = len(os.listdir(path))//2
    for i in range(n):
        matrix = np.array(Image.open(f'{path}/{i}{".0" if i else ""}Map.png'))
        Image.fromarray(bounding_square(matrix)).save(f'{path}/{i}{".0" if i else ""}MapBounded.png')
        print(f'{path}/{i}{".0" if i else ""}MapBounded.png')

dataset/0510025536_Layout1_PLAN4/0MapBounded.png
dataset/0510025536_Layout1_PLAN4/1.0MapBounded.png
dataset/0510025536_Layout1_PLAN4/2.0MapBounded.png
dataset/0510025536_Layout1_PLAN4/3.0MapBounded.png
dataset/0510025536_Layout1_PLAN4/4.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/0MapBounded.png
dataset/0510025537_Layout1_PLAN3/1.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/2.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/3.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/4.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/5.0MapBounded.png
dataset/0510025537_Layout1_PLAN3/6.0MapBounded.png
dataset/0510030965_A-40/0MapBounded.png
dataset/0510030965_A-40/1.0MapBounded.png
dataset/0510030965_A-40/2.0MapBounded.png
dataset/0510030965_A-40/3.0MapBounded.png
dataset/0510030965_A-40/4.0MapBounded.png
dataset/0510030965_A-40/5.0MapBounded.png
dataset/0510030965_A-40/6.0MapBounded.png
dataset/0510030965_A-40/7.0MapBounded.png
dataset/0510030965_A-40/8.0MapBounded.png
dataset/05100309

In [11]:
run_names = os.listdir('dataset')

for run in run_names:
    path = f'dataset/{run}'
    for x in os.listdir(path):
        if x.endswith('Bounded.png'):
            h,w = Image.open(f'dataset/{run}/{x}').size
            if h!=w: print(f'dataset/{run}/{x}', h/w)

dataset/0510035518_A_40_1_101/10.0MapBounded.png 1.0433467741935485
dataset/0510035518_A_40_1_101/11.0MapBounded.png 1.0433467741935485
dataset/0510035518_A_40_1_101/12.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/13.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/14.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/15.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/16.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/17.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/18.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/19.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/20.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/21.0MapBounded.png 1.0625
dataset/0510035518_A_40_1_101/9.0MapBounded.png 1.0443548387096775
dataset/0510035519_A_40_1_102/21.0MapBounded.png 1.0317028985507246
dataset/0510035519_A_40_1_102/22.0MapBounded.png 1.0307971014492754
dataset/0510035519_A_40_1_102/23.0MapBounded.png 1.0307971014492754
dataset/0510035519_A_40_1_102/24.